# Dataset preparation for Homework 2

This notebook prepares the only cached dataset required by the final predictive game notebook.

It does **not** perform exploratory analysis and does **not** save extra tables or plots. Its purpose is deliberately narrow:

1. read raw BTS DB1B Market CSV files from `HW_2/data/`;
2. standardize the columns needed for the project;
3. add route identifiers and a revenue proxy;
4. write one reusable parquet cache to `HW_2/cache/db1b_market_combined.parquet`.

The final model notebook reads this parquet file directly.


In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Robust project-root detection. The notebook can be executed from:
# - repository root;
# - HW_2/;
# - HW_2/notebooks/.
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "HW_2").exists():
    HW2_DIR = CURRENT_DIR / "HW_2"
elif CURRENT_DIR.name == "HW_2":
    HW2_DIR = CURRENT_DIR
elif CURRENT_DIR.parent.name == "HW_2":
    HW2_DIR = CURRENT_DIR.parent
else:
    HW2_DIR = CURRENT_DIR

DATA_DIR = HW2_DIR / "data"
CACHE_DIR = HW2_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATTERN = str(DATA_DIR / "T_DB1B_MARKET-*.csv")
COMBINED_PARQUET = CACHE_DIR / "db1b_market_combined.parquet"

print(f"Raw data directory: {DATA_DIR}")
print(f"Output cache file: {COMBINED_PARQUET}")


Raw data directory: /Users/tsarikov-i-i/4_Game_Theory&Decision_Making/HW_2/data
Output cache file: /Users/tsarikov-i-i/4_Game_Theory&Decision_Making/HW_2/cache/db1b_market_combined.parquet


## 1. Check raw input files

The project uses 16 quarterly DB1B Market files for 2021–2024. We check that they are present before building the cache.


In [2]:
csv_files = sorted(DATA_DIR.glob("T_DB1B_MARKET-*.csv"))
assert len(csv_files) == 16, f"Expected 16 quarterly CSV files, found {len(csv_files)} in {DATA_DIR}"

input_summary = pd.DataFrame({
    "metric": ["raw_csv_files", "first_file", "last_file"],
    "value": [len(csv_files), csv_files[0].name, csv_files[-1].name],
})
input_summary


,metric,value
0,raw_csv_files,16
1,first_file,T_DB1B_MARKET-2021_q1.csv
2,last_file,T_DB1B_MARKET-2024_q4.csv


## 2. Build combined parquet cache

The raw CSV files are large, so we use DuckDB to scan them efficiently and write a parquet cache.

Important details:

- `union_by_name=True` is required because 2024Q4 contains an extra `MARKET_DISTANCE` column;
- only columns needed for the project are kept;
- `ticket_revenue_proxy = passengers × market_fare` is created for later payoff calculations;
- airport-pair route identifiers are created in both directed and undirected forms;
- no fare filtering is applied here, because the final modelling notebook controls the modelling filter explicitly.


In [3]:
con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=4")
con.execute("PRAGMA memory_limit='8GB'")

raw_scan_sql = f"""
read_csv_auto(
    '{CSV_PATTERN}',
    union_by_name = true,
    filename = true,
    sample_size = 20000,
    ignore_errors = true
)
"""

# The cache is rebuilt intentionally to make this notebook reproducible.
# If the parquet file already exists, it is overwritten.
con.execute(f"""
COPY (
    SELECT
        CAST(YEAR AS INTEGER) AS year,
        CAST(QUARTER AS INTEGER) AS quarter,
        CAST(YEAR AS VARCHAR) || 'Q' || CAST(QUARTER AS VARCHAR) AS period,
        CAST(ORIGIN_AIRPORT_ID AS INTEGER) AS origin_airport_id,
        CAST(ORIGIN_CITY_MARKET_ID AS INTEGER) AS origin_city_market_id,
        CAST(ORIGIN AS VARCHAR) AS origin,
        CAST(DEST_AIRPORT_ID AS INTEGER) AS dest_airport_id,
        CAST(DEST_CITY_MARKET_ID AS INTEGER) AS dest_city_market_id,
        CAST(DEST AS VARCHAR) AS dest,
        CAST(REPORTING_CARRIER AS VARCHAR) AS carrier,
        CAST(PASSENGERS AS DOUBLE) AS passengers,
        CAST(MARKET_FARE AS DOUBLE) AS market_fare,
        TRY_CAST(MARKET_DISTANCE AS DOUBLE) AS market_distance,
        CAST(PASSENGERS AS DOUBLE) * CAST(MARKET_FARE AS DOUBLE) AS ticket_revenue_proxy,
        CAST(ORIGIN AS VARCHAR) || '-' || CAST(DEST AS VARCHAR) AS route_airport_directed,
        CASE
            WHEN CAST(ORIGIN AS VARCHAR) <= CAST(DEST AS VARCHAR)
                THEN CAST(ORIGIN AS VARCHAR) || '-' || CAST(DEST AS VARCHAR)
            ELSE CAST(DEST AS VARCHAR) || '-' || CAST(ORIGIN AS VARCHAR)
        END AS route_airport,
        CAST(ORIGIN_CITY_MARKET_ID AS VARCHAR) || '-' || CAST(DEST_CITY_MARKET_ID AS VARCHAR) AS route_city_directed,
        CASE
            WHEN CAST(ORIGIN_CITY_MARKET_ID AS INTEGER) <= CAST(DEST_CITY_MARKET_ID AS INTEGER)
                THEN CAST(ORIGIN_CITY_MARKET_ID AS VARCHAR) || '-' || CAST(DEST_CITY_MARKET_ID AS VARCHAR)
            ELSE CAST(DEST_CITY_MARKET_ID AS VARCHAR) || '-' || CAST(ORIGIN_CITY_MARKET_ID AS VARCHAR)
        END AS route_city
    FROM {raw_scan_sql}
    WHERE YEAR IS NOT NULL
      AND QUARTER IS NOT NULL
      AND REPORTING_CARRIER IS NOT NULL
      AND PASSENGERS IS NOT NULL
      AND MARKET_FARE IS NOT NULL
) TO '{COMBINED_PARQUET}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Combined parquet cache created.")


Combined parquet cache created.


## 3. Minimal validation

The validation below checks only that the cache was created and has the expected broad coverage. It is intentionally compact; detailed analysis is done in the final model notebook.


In [4]:
validation = con.execute(f"""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT period) AS quarters,
    MIN(year) AS min_year,
    MAX(year) AS max_year,
    COUNT(DISTINCT carrier) AS carriers,
    COUNT(DISTINCT route_airport) AS airport_pair_routes,
    SUM(passengers) AS passengers,
    SUM(ticket_revenue_proxy) / NULLIF(SUM(passengers), 0) AS passenger_weighted_avg_fare
FROM read_parquet('{COMBINED_PARQUET}')
""").df()

validation


,rows,quarters,min_year,max_year,carriers,airport_pair_routes,passengers,passenger_weighted_avg_fare
0,112342845,16,2021,2024,25,55765,"221,674,388.0000",207.7596


The resulting parquet file is the only prepared dataset required by `02_price_change_predictive_game.ipynb`.
